# Task 6 — Volleyball Ball Tracker 


In [2]:
import cv2 as cv
import numpy as np
import math
from collections import deque


def cal_circularity(area, perimeter):
    if perimeter == 0: 
        return 0
    return (4 * math.pi * area) / (perimeter * perimeter)


def ball_track(frame, ball_contours, height, width):
    best_center = None   # store ball center

    for cnt in ball_contours:
        area = cv.contourArea(cnt)
        if 100 < area < 350:
            x, y, w, h = cv.boundingRect(cnt)

            if y < height * 0.25:
                min_r, max_r = 4, 13
            elif y < height * 0.65:
                min_r, max_r = 8, 15
            else:
                min_r, max_r = 7, 17

            perimeter = cv.arcLength(cnt, True)
            if perimeter == 0:
                continue

            radius = max(w, h) / 2
            if not (min_r < radius < max_r):
                continue

            aspect_ratio = float(w) / h
            circularity = cal_circularity(area, perimeter)

            if 0.95 < aspect_ratio < 1.2 and 1.2 > circularity > 0.7:
                
                center = (x + w//2, y + h//2)

                best_center = center

                cv.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
                cv.circle(frame, center, int(radius), (0, 0, 255), 2)

                break

    return best_center


def count_players(frame, mask, color):
    contours, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    player_count = 0

    for contour in contours:
        area = cv.contourArea(contour)

        if area < 800:   # slightly increased threshold
            continue

        perimeter = cv.arcLength(contour, True)
        if perimeter == 0:
            continue

        circularity = cal_circularity(area, perimeter)
        x, y, w, h = cv.boundingRect(contour)
        aspect_ratio = w / float(h)

        if circularity <= 0.65 and aspect_ratio < 1.5:
            cv.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            player_count += 1

    return player_count


cap = cv.VideoCapture("Volleyball.mp4")

trajectory = deque(maxlen=7)


while True:
    ret, frame = cap.read()
    
    if not ret:
        print("End of video.")
        break

    hsv = cv.cvtColor(frame, cv.COLOR_BGR2HSV)
    height, width, _ = frame.shape    

    # BALL
    lower_yellow = np.array([12,100,200])
    upper_yellow = np.array([25,200,255])

    mask_ball = cv.inRange(hsv, lower_yellow, upper_yellow)
    
    #kernel?A small matrix used for image processing operations, Slides over the image and modifies pixels, elliptical kernel to preserve the circular geometry during morphology operrations
    kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5))
    
    mask_ball = cv.GaussianBlur(mask_ball, (5,5), 0)
    mask_ball = cv.morphologyEx(mask_ball, cv.MORPH_OPEN, kernel)
    mask_ball = cv.morphologyEx(mask_ball, cv.MORPH_CLOSE, kernel)

    ball_contours, _ = cv.findContours(mask_ball, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    current_center = ball_track(frame, ball_contours, height, width)

    # TRAJECTORY 
    trajectory.appendleft(current_center)

    for i in range(1, len(trajectory)):
        if trajectory[i-1] is None or trajectory[i] is None:
            continue

        thickness = int(np.sqrt(30/(i+1)) * 2)
        cv.line(frame, trajectory[i-1], trajectory[i], (0,255,255), thickness)

    #  PLAYERS (BLUE)
    lower_blue = np.array([110, 80, 60])
    upper_blue = np.array([145, 255, 200])

    mask_blue_player = cv.inRange(hsv, lower_blue, upper_blue)

    mask_blue_player = cv.GaussianBlur(mask_blue_player, (7,7), 0)
    mask_blue_player = cv.morphologyEx(mask_blue_player, cv.MORPH_OPEN, kernel)
    mask_blue_player = cv.morphologyEx(mask_blue_player, cv.MORPH_CLOSE, kernel)

    argentina_player_count = count_players(frame, mask_blue_player, (255, 0, 0))

    #PLAYERS (YELLOW)
    lower_yellow_player = np.array([10, 100, 100])
    upper_yellow_player = np.array([40, 255, 255])

    mask_yellow_player = cv.inRange(hsv, lower_yellow_player, upper_yellow_player)

    mask_yellow_player = cv.GaussianBlur(mask_yellow_player, (7,7), 0)
    mask_yellow_player = cv.morphologyEx(mask_yellow_player, cv.MORPH_OPEN, kernel)
    mask_yellow_player = cv.morphologyEx(mask_yellow_player, cv.MORPH_CLOSE, kernel)

    brazil_player_count = count_players(frame, mask_yellow_player, (0, 255, 255))

    # ================= DISPLAY =================
    count_text = f"BRAZIL: {brazil_player_count}  ARGENTINA: {argentina_player_count}"
    cv.putText(frame, count_text, (10, 40), cv.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 0), 2)

    cv.imshow('Ball Mask', mask_ball)
    cv.imshow('Frame', frame)

    if cv.waitKey(20) & 0xFF == ord('q'):
        break

cap.release()
cv.destroyAllWindows()

End of video.


In this notebook, we use **OpenCV** to track a volleyball in a match video.
We will:
- Detect the ball in each frame using colour detection
- Trace the ball's trajectory as it moves
- Count the number of players on each team


## Step 1 — Import Libraries


We first import all the tools we need:
- `cv2` → OpenCV, the main computer vision library
- `numpy` → for working with arrays (images are arrays)
- `matplotlib` → for displaying images inside Jupyter
- `deque` → a special list that auto-removes old items (used for ball trail)

## Define circularity function
we need a mathematical measure of how circular something is.

Formula:  circularity= 4π⋅area /perimeter^2
- ≈ 1 : perfect circle
- < 1 : irregular shape

Now , permeter should not be 0 - crash.

When can this happen?
- Very tiny contour
- Noise pixel
- Broken detection

## Define a function to count players 
A function that takes:
- frame:  original image (for drawing)
- mask : binary image (only team color visible)
- color : rectangle color (for drawing players)
  
Returns:
 number of detected players

1. A contour is the boundary (outline) of an object in an image
- cv.RETR_EXTERNAL : Contour retrieval mode- Only give me the outermost contours
- cv.CHAIN_APPROX_SIMPLE : Contour approximation method - Store only important boundary points not every pixel
- returns : contours, hierarchy

2. Initialize counter
3. Loop through contours
   - Area calculation : counts no of pixels 
   - Skip small blobs
   - Perimeter : Measures boundary length of contour
   - check p !=0
   - circularity check
   - Bounding Box
   - aspect ratio= width/ height  (Tall player< 1 , Square≈ 1, Wide objec > 1)
   - FINAL FILTER ( Not circular + Not too wide = likely a player)
   - cv.rectangle(frame, (x, y), (x + w, y + h), color, 2)
   - Increase count
4. return player count

how u computed the area threshold ?

print(area) -  Observe values - You’ll see something like: Noise(20–200) , Small blobs(200–400), Players(800–3000)

or use histogram based filtering

## BALL Tracker 
Takes all contours (from mask) & returns only valid ball-like contours

1. best center = none
2. Loop through contours
   - calculate area
   - then area filtering
   - bounding box
   - Distance-Based Radius Logic - Top of frame : ball looks smaller , Bottom : ball looks bigger
   - check p !=0
   - cal radius ands see if its within range 
   - circularity
   - aspect ratio ( for ball :  width ≈ height)
   - MAIN BALL CONDITION
   - cenetr calculate
   - store best result
   - draw circle around it 


## Tracking the ball 
Inputs:
- valid_points → list of possible balls (from previous function)
- prev_center → last known ball position
- max_distance_threshold → how far ball is allowed to move
  
Goal: From multiple possible candidates, pick the correct ball based on 

1. no valid point case
2. If we have previous position
   - Sort points by distance (which points is closest to previous ball position)
   - Pick closest
   - Calculate distance again
   - Distance threshold check
     (Previous ball at (100,100) points : A - (110,110) :(real ball) , B - (500,500) (noise)
3. If NO previous position
   - Sort by area (Ball is usually biggest among small objects)
   - Return largest

~ Nearest Neighbor Tracking (basic tracking algorithm)


## Draw Trajectory 
Inputs:
- frame : current video frame (image)
- track_points : list (deque) of previous ball positions

1. Loop through points
    - Iterates from 2nd point to last
    - Handle missing  (Sometimes ball is not detected: track_points.appendleft(None) :[(100,100), (120,110), None, (150,130)])
    - draw line
        - frame : where to draw
        - track_points[i - 1]: start point
        - track_points[i] : end point
        - (0,255,255) : color (yellow)
        - 2 : thickness

## setup / initialization layer
1. Load video
2. if path wrong it wont load ( ret : false)
3. setup / initialization layer.
4. get frame width
5. get frame height
6. define codec - video encoding format : tells how vdo will be compressed
7. Create Video Writer (Creates output video file, You can write frames into it)
8. HSV Ranges ( team 1, team 2, ball)
9. tracking points storage : stores last seven ball position
10. Previous Center
11. max distance threshold - limits how far ball can move between frames

Why deque:  fast insert/remove & auto removes old points
 
Why maxlen=7: controls tail length


## MAIN LOOP
infinte loop - 
1. Read Frame
2. when video ends get out of loop
3. blur image - to remove noise , smooth
4. convert to hsv
5. mask for team 1
6. mask for team 2
7. mask for ball
8. Morphology Setup ( ellipse - natural )
9. erosion - removes noise
10. 10 . dilation - expands object , connect broken parts
11. final erosion
12. count players ( team 1 and 2)
13. display counters
14. ball contours
15. final ball points
16. track ball
17. if ball found ?
    - update prev center as this current best center
    - draw ball
    - same position
    -  if not found : track_points.appendleft(None)
18. draw trajectory
19. show frame
20. save frame
21. exit key



DOUBTS

1. ret ? A boolean (True/False) that tells whether a frame was successfully read.
2. writter ? An object used to save processed frames into a video file
3. upper and lower ?? HSV color ranges used to detect specific colors
4. morphology ?? Operations to clean and refine binary images (masks) 
   - erosion : removes white pixel from edges
   - dilation : adds white pixels(fill gaps , reconnects broken parts)
5. Most probable issues:
   - HSV ranges → wrong masks
   - Ball filtering too strict
   - Morphology removing ball
   - Area thresholds

In [ ]:
import cv2

def get_hsv(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        print("HSV:", hsv[y, x])

cap = cv2.VideoCapture("Volleyball.mp4")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    cv2.imshow("Frame", frame)
    cv2.setMouseCallback("Frame", get_hsv)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

#use slider....
#skeletonisation...

Earlier approach:
Mask > detect > draw
Final approach:
Mask > detect > validate > track > smooth >visualize

earlier:
multiple candidates
sorting
distance threshold
rejection logic
Problem:
ball often rejected → disappears
too many conditions → fragile system

Earlier I used a multi-candidate tracking approach which was too sensitive to noise and often rejected the ball. I simplified it to a single reliable detection per frame

i detected the ball center and tried to follow it throw frames

strict candidate filtering and distance thresholds, which made it sensitive to noise and caused frequent loss of the ball.